# Using a model


In [0]:
spark.sql("USE CATALOG barbara_lakehouse")
spark.sql("USE SCHEMA ml_sandbox")

In [0]:
from pyspark.sql.functions import col
import pandas as pd

# Load dataset
data_path = "/Volumes/barbara_lakehouse/ml_sandbox/data/test.csv"
test_df = spark.read.csv(data_path, header=True, inferSchema=True)

# Cast Boolean columns to int
test_df = test_df.withColumn("PassengerId", col("PassengerId").cast("string")) \
                 .withColumn("VIP", col("VIP").cast("int")) \
                 .withColumn("CryoSleep", col("CryoSleep").cast("int"))

display(test_df)

In [0]:
%pip install mlflow

import mlflow.sklearn

# If you're running on Databricks, you may not need to set the tracking URI manually
# But if needed, do so:
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

# Define model name
model_uri = "models:/decision_tree_model/4"

# Load model
loaded_model = mlflow.sklearn.load_model(model_uri)
loaded_model

In [0]:
test = test_df.toPandas()

# Perform inference via model.predict()
predictions = loaded_model.predict(test)
predictions

In [0]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier()
model.fit([[0,0],[1,1]], [0,1])  # FAKE


In [0]:
import mlflow
import mlflow.sklearn
from sklearn.tree import DecisionTreeClassifier

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")  # IMPORTANT

model = DecisionTreeClassifier()
model.fit([[0, 0], [1, 1]], [0, 1])  # FAKE

with mlflow.start_run():
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        registered_model_name="decision_tree_model"
    )

print("✅ Fake model enregistré dans MLflow")

In [0]:
%pip install -U "mlflow-skinny[databricks]" databricks-sdk
dbutils.library.restartPython()


In [0]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

model = mlflow.sklearn.load_model("models:/decision_tree_model/1")
model


In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

model = DecisionTreeClassifier().fit([[0,0],[1,1]], [0,1])

input_example = pd.DataFrame([[0,0],[1,1]], columns=["f1","f2"])

with mlflow.start_run():
    mlflow.sklearn.log_model(
        model,
        name="model",  # remplace artifact_path
        input_example=input_example,
        registered_model_name="barbara_lakehouse.ml_sandbox.decision_tree_model"
    )
